# XGBoost and LightGBM Benchmark
Cross-sectional equity return prediction in the Emerging Markets universe using the JKP 2023 global factor panel. Both models receive the full set of rank-normalised JKP characteristics as input and are evaluated against the same six-month rebalancing, 25 basis-point transaction cost, and volatility overlay as the Dual Path Portfolio Transformer.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import xgboost as xgb
import lightgbm as lgb
import optuna
from scipy.stats import spearmanr
import matplotlib
import matplotlib.pyplot as plt

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

In [ ]:
data_path = Path('../data/Global_Factor_EM.parquet')
results_dir = Path('../results/em/tree_benchmark')
results_dir.mkdir(parents = True, exist_ok = True)

# time split: match the main pipeline fixed split
train_end = pd.Timestamp('2014-12-31')
val_end = pd.Timestamp('2019-12-31')

ret_col = 'ret_exc_lead1m'
rebalance_freq = 6
tc_bps = 25
min_stocks = 30
ret_clip_low = -1.0
ret_clip_high =  1.0

# volatility targeting: match main pipeline config
target_vol = 0.10
vol_lookback = 6
max_leverage_ls = 3.0

# hyperparameter search
n_trials_xgb = 50
n_trials_lgb = 50
optuna_seed  = 42

# last n_hpo_months of training used for hpo trials (faster)
n_hpo_months = 36

print(f'train: up to {train_end.date()}, val: to {val_end.date()}, test: remaining')

In [ ]:
schema = pq.read_schema(data_path)

non_feature = {
    'id', 'gvkey', 'isin', 'cusip', 'permno', 'permco',
    'eom', 'excntry', 'sic', 'naics', 'source_crsp', ret_col,
}
feature_cols = [
    c for c in schema.names
    if c not in non_feature
    and 'float' in str(schema.field(c).type).lower()
]

needed = list(dict.fromkeys(
    [c for c in ['id', 'eom', 'excntry', ret_col] + feature_cols
     if c in schema.names]
))

df = pd.read_parquet(data_path, columns = needed)
df['eom'] = pd.to_datetime(df['eom'])

for col in feature_cols:
    if col in df.columns and df[col].dtype == float:
        df[col] = df[col].astype(np.float32)

df[ret_col] = df[ret_col].clip(lower = ret_clip_low, upper = ret_clip_high)

print(f'loaded: {df.shape[0]:,} rows, {len(feature_cols)} characteristic columns')
print(f'date range: {df["eom"].min().date()} to {df["eom"].max().date()}')
print(f'countries: {df["excntry"].nunique()}')

In [ ]:
sorted_eoms = sorted(df['eom'].unique())
all_months = {}
n_feat = len(feature_cols)

for eom in sorted_eoms:
    month = df[df['eom'] == eom].copy()
    month = month[month[ret_col].notna()]
    if len(month) < min_stocks:
        continue

    ids = month['id'].values
    r = month[ret_col].values.astype(np.float64)

    # rank-normalise each characteristic within month to [-0.5, 0.5]
    # nan for missing values: both xgboost and lightgbm handle nan natively
    x = np.full((len(month), n_feat), np.nan, dtype = np.float32)
    for j, col in enumerate(feature_cols):
        if col not in month.columns:
            continue
        vals = month[col].values.astype(np.float64)
        valid = np.isfinite(vals)
        if valid.sum() > 1:
            x[valid, j] = (pd.Series(vals[valid]).rank(pct = True).values - 0.5
            ).astype(np.float32)

    all_months[eom] = {'ids': ids, 'r': r, 'x': x}

sorted_dates = sorted(all_months.keys())
print(f'processed: {len(sorted_dates)} months')
print(f'avg firms/month: {np.mean([len(m["ids"]) for m in all_months.values()]):.0f}')

In [ ]:
train_dates = [d for d in sorted_dates if d <= train_end]
val_dates = [d for d in sorted_dates if train_end < d <= val_end]
test_dates = [d for d in sorted_dates if d > val_end]

print(f'train:{len(train_dates)} months ({train_dates[0].date()} to {train_dates[-1].date()})')
print(f'val:{len(val_dates)} months ({val_dates[0].date()} to {val_dates[-1].date()})')
print(f'test:{len(test_dates)} months ({test_dates[0].date()} to {test_dates[-1].date()})')

# pool full training set
x_train = np.vstack([all_months[d]['x'] for d in train_dates])
y_train = np.concatenate([all_months[d]['r'] for d in train_dates])
print(f'x_train: {x_train.shape}')

# subsample last n_hpo_months for faster optuna trials
hpo_dates = train_dates[-n_hpo_months:]
x_hpo = np.vstack([all_months[d]['x'] for d in hpo_dates])
y_hpo = np.concatenate([all_months[d]['r'] for d in hpo_dates])
print(f'x_hpo: {x_hpo.shape} from {len(hpo_dates)} months (last {n_hpo_months} training months)')

In [ ]:
def portfolio_metrics(rets):
    rets = np.array(rets, dtype = np.float64)
    if len(rets) == 0:
        return {}
    tw = float((1.0 + rets).prod())
    ann_ret = -1.0 if tw <= 0 else float(tw ** (12.0 / len(rets)) - 1.0)
    ann_vol = float(rets.std() * np.sqrt(12.0))
    sharpe = ann_ret / max(ann_vol, 1e-8)
    se = float(np.sqrt((1.0 + 0.5 * sharpe ** 2) / len(rets)))
    cw = np.cumprod(1.0 + rets)
    pk = np.maximum.accumulate(cw)
    max_dd = float(((pk - cw) / pk).max()) if len(cw) > 0 else 0.0
    return {
        'ann_ret': ann_ret, 'ann_vol': ann_vol, 'sharpe': sharpe,
        'se_sharpe': se, 'max_dd': max_dd, 'n_obs': len(rets),
    }


def apply_vol_target(monthly_rets, rebalance_indices, target_vol, vol_lookback, max_leverage):
    """
    Apply volatility targeting at rebalance boundaries.

    Estimates annualised volatility from the trailing vol_lookback compounded
    period returns and applies a leverage factor for the next rebalancing
    window, matching the main pipeline convention.
    """
    scaled = np.array(monthly_rets, dtype = np.float64)
    n = len(monthly_rets)
    n_rb = len(rebalance_indices)
    period_rets = []
    for i in range(1, n_rb):
        window = np.array(monthly_rets[rebalance_indices[i - 1]:rebalance_indices[i]])
        period_rets.append(float(np.prod(1.0 + window) - 1.0))
    for i in range(n_rb):
        if i < vol_lookback:
            continue
        trailing = np.array(period_rets[max(0, i - vol_lookback):i])
        if len(trailing) < 2:
            continue
        sigma_ann = float(trailing.std() * np.sqrt(12.0 / rebalance_freq))
        lev = float(np.clip(target_vol / max(sigma_ann, 1e-8), 1.0 / max_leverage, max_leverage))
        next_rb = rebalance_indices[i + 1] if i + 1 < n_rb else n
        scaled[rebalance_indices[i]:next_rb] = (
            np.array(monthly_rets[rebalance_indices[i]:next_rb]) * lev
        )
    return scaled


def run_quintile_simulation(model, month_dates):
    """
    Run the quintile long-short simulation for a given model and date list.

    Returns monthly_rets (array), rebalance_indices (list), holdings_log (list).
    Firm-identifier-based turnover matches section 8.11 of the revision notes.
    """
    rset = set(month_dates[::rebalance_freq])
    monthly_rets = []
    rb_indices = []
    li_ids = []
    si_ids = []
    prev_li = set()
    prev_si = set()
    holdings_log = []

    for eom in month_dates:
        if eom not in all_months:
            continue
        m = all_months[eom]
        ids = m['ids']
        r = m['r']
        x = m['x']

        tcv = 0.0
        if eom in rset:
            rb_indices.append(len(monthly_rets))
            pred = model.predict(x)
            valid = np.isfinite(pred)
            if valid.sum() < 10:
                continue
            vi = ids[valid]
            vp = pred[valid]
            nq = max(1, int(len(vi) * 0.20))
            so = np.argsort(vp)
            li_ids = vi[so[::-1][:nq]].tolist()
            si_ids = vi[so[:nq]].tolist()
            li_set = set(li_ids)
            si_set = set(si_ids)
            to = (
                len(li_set - prev_li) + len(prev_li - li_set)
                + len(si_set - prev_si) + len(prev_si - si_set)
            ) / max(nq, 1)
            tcv = to * tc_bps / 10000.0
            prev_li = li_set
            prev_si = si_set
            holdings_log.append({
                'eom': str(eom), 'n_long': len(li_ids), 'n_short': len(si_ids),
            })

        if not li_ids:
            continue
        li_mask = np.isin(ids, li_ids)
        si_mask = np.isin(ids, si_ids)
        lr = r[li_mask]
        sr = r[si_mask]
        monthly_rets.append((float(lr.mean()) if len(lr) > 0 else 0.0)
            - (float(sr.mean()) if len(sr) > 0 else 0.0) - tcv
        )

    return np.array(monthly_rets), rb_indices, holdings_log

### XGBoost and LightGBM Hyperparameter Search

In [ ]:
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log = True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 5.0, log = True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 5.0, log = True),
        'tree_method': 'hist',
        'random_state': optuna_seed,
        'n_jobs': -1,
        'verbosity': 0,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(x_hpo, y_hpo)
    val_rets, val_rb_idx, _ = run_quintile_simulation(model, val_dates)
    if len(val_rets) == 0:
        return -999.0
    val_scaled = apply_vol_target(val_rets, val_rb_idx, target_vol, vol_lookback, max_leverage_ls)
    return portfolio_metrics(val_scaled).get('sharpe', -999.0)


xgb_study = optuna.create_study(
    direction = 'maximize',
    sampler = optuna.samplers.TPESampler(seed = optuna_seed),
)
xgb_study.optimize(xgb_objective, n_trials = n_trials_xgb, show_progress_bar = True)
xgb_best = xgb_study.best_params
print(f'XGBoost best val sharpe:{xgb_study.best_value:.4f}')
print(f'XGBoost best params:{xgb_best}')



def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log = True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 5.0, log = True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 5.0, log = True),
        'random_state': optuna_seed,
        'n_jobs': -1,
        'verbose': -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(x_hpo, y_hpo)
    val_rets, val_rb_idx, _ = run_quintile_simulation(model, val_dates)
    if len(val_rets) == 0:
        return -999.0
    val_scaled = apply_vol_target(val_rets, val_rb_idx, target_vol, vol_lookback, max_leverage_ls)
    return portfolio_metrics(val_scaled).get('sharpe', -999.0)


lgb_study = optuna.create_study(
    direction = 'maximize',
    sampler = optuna.samplers.TPESampler(seed = optuna_seed),
)
lgb_study.optimize(lgb_objective, n_trials = n_trials_lgb, show_progress_bar = True)
lgb_best = lgb_study.best_params
print(f'LightGBM best val sharpe:{lgb_study.best_value:.4f}')
print(f'LightGBM best params:{lgb_best}')

### Training with tined parameters

In [ ]:
xgb_model = xgb.XGBRegressor(
    **xgb_best, tree_method = 'hist',
    random_state = optuna_seed,
    n_jobs = -1, verbosity = 0,
)
xgb_model.fit(x_train, y_train)
print('XGBoost final model trained on full training set')

lgb_model = lgb.LGBMRegressor(
    **lgb_best, random_state = optuna_seed,
    n_jobs = -1, verbose = -1,
)
lgb_model.fit(x_train, y_train)
print('LightGBM final model trained on full training set')

In [ ]:
def rank_correlation_oos(model, month_dates):
    corrs = []
    for eom in month_dates:
        if eom not in all_months:
            continue
        m = all_months[eom]
        pred = model.predict(m['x'])
        valid = np.isfinite(pred) & np.isfinite(m['r'])
        if valid.sum() < 10:
            continue
        c, _ = spearmanr(pred[valid], m['r'][valid])
        if not np.isnan(c):
            corrs.append(float(c))
    return float(np.mean(corrs)) if corrs else 0.0


xgb_rc_val = rank_correlation_oos(xgb_model, val_dates)
xgb_rc_test = rank_correlation_oos(xgb_model, test_dates)
lgb_rc_val = rank_correlation_oos(lgb_model, val_dates)
lgb_rc_test = rank_correlation_oos(lgb_model, test_dates)

print(f'XGBoost  rank corr: val = {xgb_rc_val:.4f}, test = {xgb_rc_test:.4f}')
print(f'LightGBM rank corr: val = {lgb_rc_val:.4f}, test = {lgb_rc_test:.4f}')

### quintile long-short portfolio simulation on test set

In [ ]:
xgb_rets, xgb_rb_idx, xgb_holdings = run_quintile_simulation(xgb_model, test_dates)
lgb_rets, lgb_rb_idx, lgb_holdings = run_quintile_simulation(lgb_model, test_dates)

xgb_rets_scaled = apply_vol_target(xgb_rets, xgb_rb_idx, target_vol, vol_lookback, max_leverage_ls)
lgb_rets_scaled = apply_vol_target(lgb_rets, lgb_rb_idx, target_vol, vol_lookback, max_leverage_ls)

xgb_m = portfolio_metrics(xgb_rets_scaled)
lgb_m = portfolio_metrics(lgb_rets_scaled)

print(f'XGBoost  (vol-targeted): sharpe = {xgb_m["sharpe"]:.4f}, ann_ret = {xgb_m["ann_ret"] * 100:.2f}%, ann_vol = {xgb_m["ann_vol"] * 100:.2f}%')
print(f'LightGBM (vol-targeted): sharpe = {lgb_m["sharpe"]:.4f}, ann_ret = {lgb_m["ann_ret"] * 100:.2f}%, ann_vol = {lgb_m["ann_vol"] * 100:.2f}%')

In [ ]:
print('Tree Model Benchmark: EM Universe (test set)')
print(f'{"model":<20} {"rc_test":>8} {"sharpe":>8} {"se":>8} {"ann_ret":>9} {"ann_vol":>9} {"max_dd":>8} {"n_obs":>6}'
)
for name, m, rc in [
    ('xgboost',  xgb_m, xgb_rc_test),
    ('lightgbm', lgb_m, lgb_rc_test),
]:
    print(
        f'{name:<20} {rc:8.4f} {m["sharpe"]:8.4f} {m["se_sharpe"]:8.4f} {m["ann_ret"] * 100:8.2f}% {m["ann_vol"] * 100:8.2f}%'
        f' {m["max_dd"] * 100:7.2f}% {m["n_obs"]:6d}'
    )

summary = {
    'universe': 'EM',
    'n_features': len(feature_cols),
    'train_dates': {
        'start': str(train_dates[0].date()), 'end': str(train_dates[-1].date()),
        'n': len(train_dates),
    },
    'val_dates': {
        'start': str(val_dates[0].date()), 'end': str(val_dates[-1].date()),
        'n': len(val_dates),
    },
    'test_dates': {
        'start': str(test_dates[0].date()), 'end': str(test_dates[-1].date()),
        'n': len(test_dates),
    },
    'xgboost': {
        'best_params': xgb_best, 'rc_val': xgb_rc_val, 'rc_test': xgb_rc_test,
        'portfolio_metrics': xgb_m,
    },
    'lightgbm': {
        'best_params': lgb_best, 'rc_val': lgb_rc_val, 'rc_test': lgb_rc_test,
        'portfolio_metrics': lgb_m,
    },
}
with open(results_dir / 'em_tree_summary.json', 'w') as fh:
    json.dump(summary, fh, indent = 2, default = float)

np.save(results_dir / 'em_xgb_returns_scaled.npy', xgb_rets_scaled)
np.save(results_dir / 'em_lgb_returns_scaled.npy', lgb_rets_scaled)
with open(results_dir / 'em_xgb_holdings.json', 'w') as fh:
    json.dump(xgb_holdings, fh, indent = 2, default = str)
with open(results_dir / 'em_lgb_holdings.json', 'w') as fh:
    json.dump(lgb_holdings, fh, indent = 2, default = str)
print('Results saved')

In [ ]:
matplotlib.rcParams['font.size'] = 12

fig, axes = plt.subplots(2, 2, figsize = (14, 10))
fig.suptitle('XGBoost and LightGBM Benchmark: EM Universe (test set)', fontsize = 14)

# cumulative wealth
if len(xgb_rets_scaled) > 0:
    axes[0, 0].plot(np.cumprod(1.0 + xgb_rets_scaled), label = 'XGBoost', color = 'steelblue')
if len(lgb_rets_scaled) > 0:
    axes[0, 0].plot(np.cumprod(1.0 + lgb_rets_scaled), label = 'LightGBM', color = 'darkorange')
axes[0, 0].set_title('Cumulative Wealth (vol-targeted, test)')
axes[0, 0].legend()
axes[0, 0].grid(alpha = 0.3)

# sharpe bar with standard error
names = ['XGBoost', 'LightGBM']
sharpes = [xgb_m.get('sharpe', 0), lgb_m.get('sharpe', 0)]
ses = [xgb_m.get('se_sharpe', 0), lgb_m.get('se_sharpe', 0)]
axes[0, 1].bar(names, sharpes, yerr = ses, capsize = 5,
               color = ['steelblue', 'darkorange'])
axes[0, 1].set_title('Sharpe Ratios with SE (vol-targeted, test)')
axes[0, 1].grid(axis = 'y', alpha = 0.3)

# xgb optuna search history
xgb_vals = [t.value for t in xgb_study.trials if t.value is not None]
axes[1, 0].plot(np.maximum.accumulate(xgb_vals), color = 'steelblue')
axes[1, 0].scatter(range(len(xgb_vals)), xgb_vals, alpha = 0.3, s = 15, color = 'steelblue')
axes[1, 0].set_xlabel('trial')
axes[1, 0].set_ylabel('val sharpe')
axes[1, 0].set_title('XGBoost Optuna Search')
axes[1, 0].grid(alpha = 0.3)

# lgb optuna search history
lgb_vals = [t.value for t in lgb_study.trials if t.value is not None]
axes[1, 1].plot(np.maximum.accumulate(lgb_vals), color = 'darkorange')
axes[1, 1].scatter(range(len(lgb_vals)), lgb_vals, alpha = 0.3, s = 15, color = 'darkorange')
axes[1, 1].set_xlabel('trial')
axes[1, 1].set_ylabel('val sharpe')
axes[1, 1].set_title('LightGBM Optuna Search')
axes[1, 1].grid(alpha = 0.3)

plt.tight_layout()
plt.savefig(results_dir / 'em_tree_benchmark.png', dpi = 150, bbox_inches = 'tight')
plt.show()
print(f'plot saved: {results_dir / "em_tree_benchmark.png"}')